In [1]:
!pip install opencv-python pandas numpy tqdm

In [3]:
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm

video_path = "interview.mp4"

cap = cv2.VideoCapture(video_path)

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

movement_data = []
frame_num = 1

for i in tqdm(range(frame_count)):

    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1

    if frame_num % 5 != 0:
        continue

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    diff = cv2.absdiff(prev_gray, gray)

    movement_score = np.sum(diff)

    movement_data.append({
        "frame": frame_num,
        "movement_score": movement_score
    })

    prev_gray = gray

cap.release()

movement_df = pd.DataFrame(movement_data)

print(movement_df.head())

movement_df.to_csv("body_movement.csv", index=False)

100%|█████████▉| 53901/53902 [05:12<00:00, 172.34it/s]


   frame  movement_score
0      5         2193430
1     10         2533373
2     15         2696415
3     20         2642536
4     25         2576884


In [5]:
import cv2
import pandas as pd
from tqdm import tqdm

video_path = "interview.mp4"

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_smile.xml")

cap = cv2.VideoCapture(video_path)

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

data = []
frame_num = 0

for i in tqdm(range(frame_count)):

    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1

    if frame_num % 5 != 0:
        continue

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    expression = "neutral"

    for (x, y, w, h) in faces:

        roi_gray = gray[y:y+h, x:x+w]

        smiles = smile_cascade.detectMultiScale(roi_gray, 1.7, 20)

        if len(smiles) > 0:
            expression = "smile"
        else:
            expression = "neutral"

        break

    data.append({
        "frame": frame_num,
        "expression": expression
    })

cap.release()

expression_df = pd.DataFrame(data)

print(expression_df.head())

expression_df.to_csv("facial_expression.csv", index=False)

100%|██████████| 53902/53902 [15:05<00:00, 59.52it/s]

   frame expression
0      5    neutral
1     10    neutral
2     15    neutral
3     20    neutral
4     25    neutral
